# Benchmark Results Table

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
OUR_METHOD  = 'normal'
UCI_EXPERIMENTS = ['energy', 'kin8nm', 'yacht']
PDE_EXPERIMENTS = ['Burgers', 'KS']                                    # distributional_method to select for our model
EXPERIMENTS = UCI_EXPERIMENTS.copy()
EXPERIMENTS.extend(PDE_EXPERIMENTS)
METRICS     = ['RMSE', 'CRPS']                           # sub-columns

import os

# Find the results directory regardless of where the notebook is launched from
_candidates = [
    os.path.join(os.getcwd(), 'results'),         # running from project root
    os.path.join(os.getcwd(), '..', 'results'),   # running from evaluation/
]
BASE_PATH = next(
    (os.path.abspath(p) for p in _candidates if os.path.isdir(p)),
    os.path.abspath(os.path.join(os.getcwd(), 'results')),  # fallback
)
print('Results base path:', BASE_PATH)

Results base path: /home/math/buelte/projects/diff/results


In [2]:
import pandas as pd
import numpy as np
from glob import glob


def load_csv_as_runs(path):
    """Read a transposed test.csv (rows=params, cols=runs) and return (runs × params) DataFrame."""
    df = pd.read_csv(path, index_col=0)
    return df.T.reset_index(drop=True)


def mean_metric(df, metric_col, distrib_filter=None):
    """Return the mean of *metric_col*, optionally filtering by distributional_method."""
    if distrib_filter is not None and 'distributional_method' in df.columns:
        df = df[df['distributional_method'] == distrib_filter]
    if df.empty or metric_col not in df.columns:
        return np.nan
    return df[metric_col].astype(float).mean()


def load_pde_result(experiment, method):
    """Load Burgers / KS results for a given distributional method."""
    path = os.path.join(BASE_PATH, experiment, method, 'test.csv')
    if not os.path.exists(path):
        return np.nan, np.nan
    df = load_csv_as_runs(path)
    return mean_metric(df, 'RMSETest'), mean_metric(df, 'CRPSTest')


def load_uci_result(experiment, method=None, subfolder=None):
    """
    Load UCI results.
    - subfolder=None         → results/crps_ensemble/<experiment>/test.csv
    - subfolder='UCI'        → results/UCI/<experiment>/*/test.csv  (filtered by method)
    - subfolder='NDP'        → results/NDP/<experiment>/test.csv
    """
    if subfolder is None:
        paths = [os.path.join(BASE_PATH, 'crps_ensemble', experiment, 'test.csv')]
    elif subfolder == 'NDP':
        paths = [os.path.join(BASE_PATH, 'NDP', experiment, 'test.csv')]
    else:
        paths = glob(os.path.join(BASE_PATH, subfolder, experiment, '*', 'test.csv'))

    dfs = []
    for p in paths:
        if os.path.exists(p):
            dfs.append(load_csv_as_runs(p))

    if not dfs:
        return np.nan, np.nan

    combined = pd.concat(dfs, ignore_index=True)
    return mean_metric(combined, 'RMSETest', method), mean_metric(combined, 'CRPSTest', method)

In [3]:
# ── Collect results ────────────────────────────────────────────────────────────
rows = {}

# ── Our method ────────────────────────────────────────────────────────────────
our_row = {}
for exp in PDE_EXPERIMENTS:
    rmse, crps = load_pde_result(exp, OUR_METHOD)
    our_row[(exp, 'RMSE')] = rmse
    our_row[(exp, 'CRPS')] = crps
for exp in UCI_EXPERIMENTS:
    rmse, crps = load_uci_result(exp, method=OUR_METHOD, subfolder='UCI')
    our_row[(exp, 'RMSE')] = rmse
    our_row[(exp, 'CRPS')] = crps
rows[f'Ours ({OUR_METHOD})'] = our_row

# ── CRPS-Ensemble ─────────────────────────────────────────────────────────────
crps_ens_row = {}
for exp in PDE_EXPERIMENTS:
    rmse, crps = load_uci_result(exp, subfolder=None)   # crps_ensemble folder
    crps_ens_row[(exp, 'RMSE')] = rmse
    crps_ens_row[(exp, 'CRPS')] = crps
for exp in UCI_EXPERIMENTS:
    rmse, crps = load_uci_result(exp, subfolder=None)
    crps_ens_row[(exp, 'RMSE')] = rmse
    crps_ens_row[(exp, 'CRPS')] = crps
rows['CRPS-Ensemble'] = crps_ens_row

# ── NDP ───────────────────────────────────────────────────────────────────────
ndp_row = {}
for exp in PDE_EXPERIMENTS:
    rmse, crps = load_uci_result(exp, subfolder='NDP')
    ndp_row[(exp, 'RMSE')] = rmse
    ndp_row[(exp, 'CRPS')] = crps
for exp in UCI_EXPERIMENTS:
    # NDP has no UCI results yet
    ndp_row[(exp, 'RMSE')] = np.nan
    ndp_row[(exp, 'CRPS')] = np.nan
rows['NDP'] = ndp_row

print('Results collected.')

Results collected.


In [4]:
# ── Build multi-index DataFrame ────────────────────────────────────────────────
SCALE = {'Burgers': 1000, 'KS': 100}  # scale factors per experiment

col_index = pd.MultiIndex.from_product([EXPERIMENTS, METRICS],
                                        names=['Experiment', 'Metric'])

df_table = pd.DataFrame(
    index=pd.Index(rows.keys(), name='Method'),
    columns=col_index,
    dtype=float
)

for method, row in rows.items():
    for (exp, metric), val in row.items():
        df_table.loc[method, (exp, metric)] = val * SCALE.get(exp, 1)

df_table

Experiment       energy              kin8nm               yacht            \
Metric             RMSE      CRPS      RMSE      CRPS      RMSE      CRPS   
Method                                                                      
Ours (normal)  0.453168  0.244401  0.068832  0.038112  0.906206  0.347457   
CRPS-Ensemble  0.502862  0.283139  0.079461  0.047035  1.033802  0.484462   
NDP                 NaN       NaN       NaN       NaN       NaN       NaN   

Experiment      Burgers                  KS            
Metric             RMSE      CRPS      RMSE      CRPS  
Method                                                 
Ours (normal)  0.807800  0.120260  0.390332  0.210220  
CRPS-Ensemble  1.018167  0.142633  0.584423  0.329573  
NDP            1.339900  0.302200  5.333440  2.555920

In [5]:
# ── Formatted display (2 decimal places, NaN → dash) ──────────────────────────
df_display = df_table.map(lambda x: f'{x:.2f}' if pd.notna(x) else '--')
df_display

Experiment    energy       kin8nm       yacht       Burgers          KS      
Metric          RMSE  CRPS   RMSE  CRPS  RMSE  CRPS    RMSE  CRPS  RMSE  CRPS
Method                                                                       
Ours (normal)   0.45  0.24   0.07  0.04  0.91  0.35    0.81  0.12  0.39  0.21
CRPS-Ensemble   0.50  0.28   0.08  0.05  1.03  0.48    1.02  0.14  0.58  0.33
NDP               --    --     --    --    --    --    1.34  0.30  5.33  2.56

In [6]:
# ── LaTeX output ──────────────────────────────────────────────────────────────
latex_str = df_display.to_latex(
    multicolumn=True,
    multicolumn_format='c',
    multirow=True,
    escape=False,
    caption='Benchmark results (mean across seeds). -- indicates unavailable results.',
    label='tab:benchmark_results',
    position='htbp',
)
print(latex_str)

\begin{table}[htbp]
\caption{Benchmark results (mean across seeds). -- indicates unavailable results.}
\label{tab:benchmark_results}
\begin{tabular}{lllllllllll}
\toprule
Experiment & \multicolumn{2}{c}{energy} & \multicolumn{2}{c}{kin8nm} & \multicolumn{2}{c}{yacht} & \multicolumn{2}{c}{Burgers} & \multicolumn{2}{c}{KS} \\
Metric & RMSE & CRPS & RMSE & CRPS & RMSE & CRPS & RMSE & CRPS & RMSE & CRPS \\
Method &  &  &  &  &  &  &  &  &  &  \\
\midrule
Ours (normal) & 0.45 & 0.24 & 0.07 & 0.04 & 0.91 & 0.35 & 0.81 & 0.12 & 0.39 & 0.21 \\
CRPS-Ensemble & 0.50 & 0.28 & 0.08 & 0.05 & 1.03 & 0.48 & 1.02 & 0.14 & 0.58 & 0.33 \\
NDP & -- & -- & -- & -- & -- & -- & 1.34 & 0.30 & 5.33 & 2.56 \\
\bottomrule
\end{tabular}
\end{table}

